# Jev_CeNN — **Bonsai-inspired ternary g128, one layer at a time**

**Source model:** [`vtava/Jev_CeNN`](https://huggingface.co/vtava/Jev_CeNN), a *private* standalone V31 typed-decision ModernBERT encoder (30 encoder blocks). This notebook uses the model's **own** `bundle.json`, `runtime.py` and `load_model.py`. No generic `AutoModelForCausalLM` assumption.

**Experiment:** For each encoder block, collect calibration statistics → ternarize candidate `nn.Linear` modules to **{-1, 0, +1} with FP16 scale per 128 input weights** → pack four codes per byte → run a **real Triton CUDA matmul** that decodes packed codes → test cumulative decision agreement / Jensen-Shannon divergence → **commit or revert**. Try all eligible projections, only attention, only feed-forward, and the largest eligible projection. Results are written after each layer.

**Important distinctions:**
- This uses a **Bonsai-like weight format**, *not* PrismML's undisclosed/proprietary training algorithm or a native GGUF file. Direct post-training quantization may seriously reduce accuracy.
- 3-way weight coding has an entropy floor of log2(3) ≈ 1.585 bits, but the *deployed format here uses 2 bits + one 16-bit scale / 128 weights* = **2.125 bits/weight**, excluding bias, padding, and other FP weights.
- Report **real GPU wall-clock latency**; packed storage is not evidence of a speedup. Torch/ModernBERT overhead can dominate when only a few layers change.
- Selection and audit use **disjoint cases from the training split**; official test is only evaluated *after* layer selection. Dev-based fidelity is not a production accuracy guarantee.
- The original private Hugging Face model is **never modified**. Upload is disabled by default.

Set **Runtime → Change runtime type → T4/L4 GPU**. L4 preferred for benchmarking. Add a Hugging Face read/write token under Colab Secrets named `HF_TOKEN` (write permission only if you explicitly choose to publish).

In [ ]:
#@title 1. Install pinned model dependencies
%pip -q install "transformers==4.57.6" "huggingface_hub>=0.25,<1" "safetensors>=0.4" "datasets>=2.20,<5" "triton>=3.0,<4" pandas matplotlib

In [ ]:
#@title 2. Configuration, private Hugging Face authentication, model download
import gc, os, sys, time, json, math, random, shutil, importlib.util
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from huggingface_hub import snapshot_download
from safetensors.torch import save_file
from IPython.display import display

SOURCE_REPO = 'vtava/Jev_CeNN' #@param {type:'string'}
OUTPUT_REPO = 'vtava/Jev_CeNN_Bonsai_G128' #@param {type:'string'}
PUBLISH_TO_HF = False #@param {type:'boolean'}
INPUT_CASES = 220 #@param {type:'integer'}
CALIBRATION_ITEMS = 20 #@param {type:'integer'}
SELECTION_ITEMS = 64 #@param {type:'integer'}
AUDIT_ITEMS = 64 #@param {type:'integer'}
MAX_LEN = 256 #@param {type:'integer'}
SELECTION_BATCH_SIZE = 2 #@param {type:'integer'}
MIN_PROJECTION_ELEMENTS = 4096 #@param {type:'integer'}
MIN_AGREEMENT = 0.985 #@param {type:'number'}
MAX_MEAN_JS = 0.012 #@param {type:'number'}
MAX_GOLD_ACCURACY_DROP = 0.015 #@param {type:'number'}
CALIBRATION_METHOD = 'activation_mse' #@param ['activation_mse', 'mse', 'naive']
LAYER_ORDER = 'forward' #@param ['forward', 'reverse']
FIRST_LAYER = 0 #@param {type:'integer'}
LAST_LAYER_EXCLUSIVE = 30 #@param {type:'integer'}
EVALUATE_OFFICIAL_TEST = True #@param {type:'boolean'}
BENCH_REPEATS = 8 #@param {type:'integer'}
SEED=2030
OUTPUT=Path('/content/Jev_CeNN_Bonsai_G128')
OUTPUT.mkdir(parents=True,exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
assert torch.cuda.is_available(), 'Select GPU in Colab Runtime settings.'
assert torch.cuda.get_device_capability()[0]>=7, 'Requires T4/L4 or newer tensor-core GPU.'
torch.backends.cuda.matmul.allow_tf32=False
try:
    from google.colab import userdata
    HF_TOKEN=userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN=os.getenv('HF_TOKEN')
if not HF_TOKEN:
    from huggingface_hub import login
    login()
    HF_TOKEN=None
ROOT=Path(snapshot_download(repo_id=SOURCE_REPO,token=HF_TOKEN))
manifest=json.loads((ROOT/'bundle.json').read_text())
assert manifest['architecture']=='laya-v31-all-attention', manifest
assert (ROOT/'runtime.py').is_file() and (ROOT/'load_model.py').is_file()
assert (ROOT/'model.safetensors').is_file()
print('GPU:',torch.cuda.get_device_name(0),'| source:',ROOT)
print('Manifest:',{k:manifest.get(k) for k in ['architecture','global_layers','sliding_layers','max_len']})

In [ ]:
#@title 3. Self-contained ternary g128 CUDA runtime
# Writes exact source into Colab so the exported bundle is reproducible.
Path('/content/jev_bonsai_g128_runtime.py').write_text('"""Bonsai-inspired g128 ternary linear modules for Jev_CeNN experimental ablation.\n\nThe format is compatible in principle with Q2_0-style ternary coding: 2-bit\ncodes 0/1/2 representing -1/0/+1 and one fp16 group scale per 128 weights.\nIt is NOT the proprietary Bonsai training method or a llama.cpp GGUF loader.\n"""\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nGROUP_SIZE = 128\n\n\ndef fit_g128(weight, importance=None, method=\'activation_mse\', search=(0.85,1.0,1.15,1.35,1.6,1.9), iterations=3):\n    """Return row-major ternary signed int8 codes (O,Kpad), scales (O,G), and weighted error.\n\n    Search scale initializations and alternating LS scale/ternary-code refits.\n    Importance is diagonal E[x_i^2] estimated from calibration data;\n    this approximates output reconstruction under uncorrelated activations.\n    """\n    assert weight.ndim == 2\n    o,k=weight.shape\n    padded=((k+GROUP_SIZE-1)//GROUP_SIZE)*GROUP_SIZE\n    w=F.pad(weight.detach().float(),(0,padded-k)).reshape(o,-1,GROUP_SIZE)\n    if importance is None or method==\'mse\':\n        imp=torch.ones((1,1,GROUP_SIZE),device=weight.device,dtype=torch.float32).expand(1,w.shape[1],-1).clone()\n    else:\n        if importance.numel()!=k:raise ValueError(\'Calibration feature mismatch\')\n        imp=F.pad(importance.detach().to(weight.device,dtype=torch.float32).clamp_min(0),(0,padded-k)).reshape(1,-1,GROUP_SIZE)\n        imp=imp / imp.mean(-1,keepdim=True).clamp_min(1e-9)\n    if padded>k:\n        last_valid=k-(padded-GROUP_SIZE)\n        imp=imp.clone()\n        imp[:,-1,last_valid:]=0\n    imp=imp.clamp_min(0)\n    mean_abs=w.abs().mean(-1).clamp_min(1e-7)\n    best_err=torch.full(mean_abs.shape,float(\'inf\'),dtype=torch.float32,device=w.device)\n    best_scale=torch.ones_like(mean_abs)\n    best_q=torch.zeros_like(w,dtype=torch.int8)\n    factors=search if method != \'naive\' else (1.5,)\n    for factor in factors:\n        scale=(mean_abs*factor).clamp_min(1e-7)\n        for _ in range(iterations if method!=\'naive\' else 1):\n            q=(w / scale.unsqueeze(-1)).round().clamp(-1,1)\n            numerator=(imp*w*q).sum(-1)\n            denominator=(imp*q.square()).sum(-1).clamp_min(1e-9)\n            scale=(numerator/denominator).clamp_min(1e-7)\n        q=(w/scale.unsqueeze(-1)).round().clamp(-1,1)\n        err=(imp*(w-q*scale.unsqueeze(-1)).square()).sum(-1)\n        take=err<best_err\n        best_err=torch.where(take,err,best_err)\n        best_scale=torch.where(take,scale,best_scale)\n        best_q=torch.where(take.unsqueeze(-1),q.to(torch.int8),best_q)\n    return best_q.reshape(o,padded), best_scale.to(torch.float16).contiguous(), float(best_err.sum().item())\n\n\ndef pack(q):\n    """Ternary int8 values -1/0/+1 -> 2-bit slots, 4 per byte."""\n    if q.ndim!=2 or q.shape[1]%4:raise ValueError(\'Expected 2D and K divisible by 4\')\n    if not bool(((q>=-1)&(q<=1)).all()):raise ValueError(\'Not ternary\')\n    c=(q.to(torch.int16)+1).reshape(q.shape[0],-1,4)\n    return (c[:,:,0] | (c[:,:,1]<<2) | (c[:,:,2]<<4) | (c[:,:,3]<<6)).to(torch.uint8).contiguous()\n\n\ndef unpack(packed, kpad):\n    p=packed.to(torch.int16)\n    return (torch.stack([(p>>s)&3 for s in (0,2,4,6)],dim=-1).reshape(p.shape[0],-1)[:,:kpad]-1).to(torch.int8)\n\ntry:\n    import triton\n    import triton.language as tl\n    TRITON_AVAILABLE=True\nexcept ImportError:\n    TRITON_AVAILABLE=False\n\nif TRITON_AVAILABLE:\n    @triton.jit\n    def _g128_kernel(X,P,S,B,Y,M:tl.constexpr,N:tl.constexpr,K:tl.constexpr,\n                     KP:tl.constexpr,G:tl.constexpr,HAS_BIAS:tl.constexpr,\n                     BM:tl.constexpr,BN:tl.constexpr):\n        rows=tl.program_id(0)*BM+tl.arange(0,BM)\n        cols=tl.program_id(1)*BN+tl.arange(0,BN)\n        offsets=tl.arange(0,128)\n        acc=tl.full((BM,BN),0.0,tl.float32)\n        for g in range(tl.cdiv(K,128)):\n            kk=g*128+offsets\n            xx=tl.load(X+rows[:,None]*K+kk[None,:],(rows[:,None]<M)&(kk[None,:]<K),0)\n            pp=tl.load(P+cols[None,:]*KP+kk[:,None]//4,\n                       (cols[None,:]<N)&(kk[:,None]<K),0x55)\n            cc=((pp.to(tl.int32)>>((kk[:,None]%4)*2))&3)-1\n            dot=tl.dot(xx,cc.to(xx.dtype))\n            scale=tl.load(S+cols*G+g,cols<N,0).to(tl.float32)\n            acc += dot.to(tl.float32)*scale[None,:]\n        if HAS_BIAS:\n            acc += tl.load(B+cols,cols<N,0).to(tl.float32)[None,:]\n        tl.store(Y+rows[:,None]*N+cols[None,:],acc,(rows[:,None]<M)&(cols[None,:]<N))\n\n\nclass TernaryG128Linear(nn.Module):\n    """Real packed on-device weights (O,Kpad/4) and fp16 scale (O,ceil(K/128)).\n\n    CUDA uses a direct Triton matrix multiply that decodes 2-bit codes in the\n    kernel without allocating FP16 reconstructed weights. CPU has a reference.\n    """\n    def __init__(self,in_features,out_features,bias=False,device=\'cpu\'):\n        super().__init__()\n        self.in_features=int(in_features)\n        self.out_features=int(out_features)\n        self.kpad=((self.in_features+127)//128)*128\n        self.groups=self.kpad//128\n        self.register_buffer(\'packed_weight\',torch.empty((self.out_features,self.kpad//4),device=device,dtype=torch.uint8))\n        self.register_buffer(\'group_scales\',torch.ones((self.out_features,self.groups),device=device,dtype=torch.float16))\n        self.register_buffer(\'bias\',torch.zeros(self.out_features,device=device,dtype=torch.float16) if bias else None)\n\n    @classmethod\n    @torch.no_grad()\n    def from_linear(cls,module,importance=None,method=\'activation_mse\'):\n        assert isinstance(module,nn.Linear)\n        result=cls(module.in_features,module.out_features,module.bias is not None,device=module.weight.device)\n        q,s,_=fit_g128(module.weight,importance=importance,method=method)\n        result.packed_weight.copy_(pack(q))\n        result.group_scales.copy_(s)\n        if module.bias is not None:result.bias.copy_(module.bias.detach().to(torch.float16))\n        return result\n\n    def unpack_weight(self):\n        q=unpack(self.packed_weight,self.kpad).reshape(self.out_features,self.groups,128).float()\n        return (q*self.group_scales.float().unsqueeze(-1)).reshape(self.out_features,self.kpad)[:,:self.in_features]\n\n    def forward(self,x):\n        if not x.is_cuda:\n            w=self.unpack_weight().to(x.dtype)\n            return F.linear(x,w,self.bias.to(x.dtype) if self.bias is not None else None)\n        if not TRITON_AVAILABLE:raise RuntimeError(\'CUDA ternary operation requires Triton\')\n        original_dtype=x.dtype\n        xc=x.contiguous().reshape(-1,self.in_features)\n        if xc.dtype not in (torch.bfloat16,torch.float16):\n            xc=xc.to(torch.float16)\n        m=xc.shape[0]\n        out=torch.empty((m,self.out_features),device=xc.device,dtype=xc.dtype)\n        bm=16 if m<32 else 32\n        bn=32 if self.out_features<64 else 64\n        _g128_kernel[(triton.cdiv(m,bm),triton.cdiv(self.out_features,bn))](\n            xc,self.packed_weight,self.group_scales,\n            self.bias if self.bias is not None else self.group_scales,out,\n            m,self.out_features,self.in_features,self.kpad//4,self.groups,\n            self.bias is not None,bm,bn,num_warps=4)\n        result=out.reshape(*x.shape[:-1],self.out_features)\n        if original_dtype==torch.float32 and not torch.is_autocast_enabled(\'cuda\'):\n            return result.float()\n        return result\n\n\ndef state_bytes(model):\n    return sum(t.numel()*t.element_size() for t in model.state_dict().values())\n\n\ndef get_parent(model,name):\n    parent,_,attr=name.rpartition(\'.\')\n    return (model.get_submodule(parent) if parent else model),attr\n\n\ndef swap(model,name,replacement):\n    parent,attr=get_parent(model,name)\n    previous=getattr(parent,attr)\n    setattr(parent,attr,replacement)\n    return previous\n', encoding='utf-8')
print('Ternary runtime written:',len(Path('/content/jev_bonsai_g128_runtime.py').read_text()),'bytes')

In [ ]:
#@title 4. Standalone packed-bundle loader
Path('/content/jev_bonsai_g128_loader.py').write_text('"""Standalone loader for the Jev_CeNN Bonsai-inspired g128 packed bundle.\nReview bundled Python before loading. The source model has a custom runtime.\n"""\nfrom pathlib import Path\nimport importlib.util\nimport json\nimport sys\n\n\ndef _import(name, file):\n    spec=importlib.util.spec_from_file_location(name,str(file))\n    if spec is None or spec.loader is None:raise ImportError(f\'Cannot import {file}\')\n    mod=importlib.util.module_from_spec(spec)\n    sys.modules[name]=mod\n    spec.loader.exec_module(mod)\n    return mod\n\n\ndef load_bonsai(path_or_repo, device=\'cuda\', token=None, revision=None):\n    import torch\n    from huggingface_hub import snapshot_download\n    from safetensors.torch import load_file\n    from transformers import AutoTokenizer\n    root=Path(str(path_or_repo))\n    if not (root.is_dir() and (root/\'bonsai_meta.json\').is_file()):\n        root=Path(snapshot_download(repo_id=str(path_or_repo), token=token, revision=revision))\n    manifest=json.loads((root/\'bundle.json\').read_text())\n    meta=json.loads((root/\'bonsai_meta.json\').read_text())\n    assert manifest.get(\'architecture\') == \'laya-v31-all-attention\'\n    assert meta.get(\'format\') == \'bonsai_inspired_q2_g128_v1\'\n    base_runtime=_import(\'_bonsai_runtime_base\',root/\'runtime.py\')\n    qr=_import(\'_bonsai_runtime_ops\',root/\'jev_bonsai_g128_runtime.py\')\n    model=base_runtime.build_model(root,manifest).to(dtype=torch.float16)\n    for spec in meta[\'quantized_modules\']:\n        name=spec[\'name\']\n        old=model.get_submodule(name)\n        if not isinstance(old,torch.nn.Linear):raise TypeError(f\'Expected linear at {name}\')\n        new=qr.TernaryG128Linear(old.in_features,old.out_features,old.bias is not None)\n        qr.swap(model,name,new)\n    weights=load_file(str(root/\'model_bonsai.safetensors\'),device=\'cpu\')\n    model.load_state_dict(weights,strict=True)\n    model.to(device=device).eval().requires_grad_(False)\n    tokenizer=AutoTokenizer.from_pretrained(str(root/\'tokenizer\'))\n    return model,tokenizer,base_runtime\n', encoding='utf-8')
print('Portable loader written')

In [ ]:
#@title 5. Load original Jev_CeNN, audit eligible layer projections, check CUDA kernel
import importlib
sys.path.insert(0,'/content')
import jev_bonsai_g128_runtime as g128
importlib.reload(g128)
ref_spec=importlib.util.spec_from_file_location('_jev_original_loader',ROOT/'load_model.py')
original_loader=importlib.util.module_from_spec(ref_spec)
ref_spec.loader.exec_module(original_loader)
model,tokenizer=original_loader.load_model(ROOT,device='cpu')
model.to(device='cuda',dtype=torch.float16).eval().requires_grad_(False)
assert len(model.encoder.layers)==30, 'Unexpected checkpoint layer count; inspect architecture.'
LAYER_PATHS=[]
for i in range(len(model.encoder.layers)):
    layer_prefix=f'encoder.layers.{i}.'
    eligible=[(name,sub) for name,sub in model.named_modules() if
        name.startswith(layer_prefix) and isinstance(sub,nn.Linear) and
        sub.weight.numel()>=MIN_PROJECTION_ELEMENTS]
    LAYER_PATHS.append([name for name,_ in eligible])
    kinds={'attention':sum('.attn.' in n for n,_ in eligible),
           'feed_forward':sum('.attn.' not in n for n,_ in eligible)}
    print(f'layer {i:02d}: {len(eligible):2d} linear matrices | {kinds}')
assert any(LAYER_PATHS), 'No eligible encoder nn.Linear projections found.'
TOTAL_PARAMS=sum(p.numel() for p in model.parameters())
ORIGINAL_STATE_BYTES=g128.state_bytes(model)
ALL_ELIGIBLE=sum(model.get_submodule(n).weight.numel() for group in LAYER_PATHS for n in group)
print('Parameters:',TOTAL_PARAMS,'| eligible:',ALL_ELIGIBLE, '| state MiB:',round(ORIGINAL_STATE_BYTES/2**20,1))

# Test actual CUDA math, not just pack/unpack.
probe=nn.Linear(256,128,bias=True,device='cuda',dtype=torch.float16)
quant=g128.TernaryG128Linear.from_linear(probe,torch.rand(256,device='cuda')+0.2)
q,sc,_=g128.fit_g128(probe.weight,torch.rand(256,device='cuda')+0.2)
assert torch.equal(g128.unpack(g128.pack(q),q.shape[1]),q)
x=torch.randn((32,256),device='cuda',dtype=torch.float16)
y_gpu=quant(x)
y_ref=F.linear(x,quant.unpack_weight().half(),quant.bias)
err=(y_gpu.float()-y_ref.float()).abs()
print('GPU packed kernel max abs:',float(err.max()),'mean abs:',float(err.mean()))
torch.testing.assert_close(y_gpu,y_ref,atol=0.03,rtol=0.02)
print('Packed Triton CUDA kernel correctness: PASS')
del probe,quant,x,y_gpu,y_ref,q,sc,err
gc.collect();torch.cuda.empty_cache()

In [ ]:
#@title 6. Build disjoint case-level calibration/selection/audit datasets
from datasets import load_dataset
rt_spec=importlib.util.spec_from_file_location('_jev_runtime',ROOT/'runtime.py')
original_runtime=importlib.util.module_from_spec(rt_spec)
rt_spec.loader.exec_module(original_runtime)
MAX_LEN=min(MAX_LEN,int(manifest.get('max_len',MAX_LEN)))

def jsonish(v):
    if isinstance(v,str):
        try:return json.loads(v)
        except Exception:return v
    return v

def canon_label(x):
    s=str(x).strip().lower()
    try:
        f=float(s)
        if math.isfinite(f) and f.is_integer():return str(int(f))
    except Exception:pass
    return s

def infer_gold(q,answer):
    if not isinstance(answer,dict):return None
    probs=jsonish(answer.get('probabilities',{}))
    if not isinstance(probs,dict):return None
    if q['t']=='choice':labels=[str(k) for k in q['crit']]
    elif q['t']=='noul':labels=['false','true']
    else:
        labels=[str(k) for k in probs]
        if all(canon_label(k).lstrip('-').isdigit() for k in labels):
            labels.sort(key=lambda k:int(canon_label(k)))
    label=canon_label(answer.get('label',max(probs,key=probs.get)))
    canonical=[canon_label(t) for t in labels]
    return canonical.index(label) if label in canonical else None

def items_from_rows(rows):
    result=[]
    for row in rows:
        state=jsonish(row['state']);questions=jsonish(row['questions']);gold=jsonish(row.get('gold',{}))
        if not isinstance(questions,dict):continue
        for qid,qdef in questions.items():
            try:
                q=original_runtime.normalize_question(qdef)
                ids,marks=original_runtime.build_sequence(
                    tokenizer,state,q,max_len=MAX_LEN,
                    head_max_len=192,truncate_left=isinstance(state,list))
                if len(marks)<2:continue
                g=infer_gold(q,gold.get(qid)) if isinstance(gold,dict) else None
                if g is not None and g>=len(marks):g=None
                result.append(dict(ids=ids,markers=marks,qtype=original_runtime.QTYPES[q['t']],gold=g))
            except (TypeError,ValueError,KeyError,IndexError):continue
    return result

# These three groups NEVER reuse a source case, even when a case has many questions.
rows=list(load_dataset('LocalLLaMA/typed-decisions','all',split=f'train[:{INPUT_CASES}]'))
assert len(rows)>=90,'Too few real training cases for separate data partitions.'
random.Random(SEED).shuffle(rows)
a=int(len(rows)*0.20);b=int(len(rows)*0.60)
cal_items=items_from_rows(rows[:a]);selection_items=items_from_rows(rows[a:b]);audit_items=items_from_rows(rows[b:])
assert len(cal_items)>=CALIBRATION_ITEMS,'Not enough calibration examples.'
assert len(selection_items)>=SELECTION_ITEMS,'Not enough selection examples.'
assert len(audit_items)>=AUDIT_ITEMS,'Not enough audit examples.'
cal_items=cal_items[:CALIBRATION_ITEMS]
selection_items=selection_items[:SELECTION_ITEMS]
audit_items=audit_items[:AUDIT_ITEMS]

def make_batch(selected,pad_to=None):
    t=pad_to or min(MAX_LEN,max(len(z['ids']) for z in selected))
    k=max(len(z['markers']) for z in selected)
    ip,am,mp,mm,qt=[],[],[],[],[]
    for z in selected:
        ids=z['ids'][:t]
        pos=[p for p in z['markers'] if p<len(ids)]
        if len(pos)<2:raise ValueError('Truncation removed answer markers')
        ip.append(ids+[int(tokenizer.pad_token_id)]*(t-len(ids)))
        am.append([1]*len(ids)+[0]*(t-len(ids)))
        mp.append(pos+[0]*(k-len(pos)))
        mm.append([1]*len(pos)+[0]*(k-len(pos)))
        qt.append(z['qtype'])
    return {key:torch.tensor(value,device='cuda',dtype=dtype) for key,value,dtype in [
        ('input_ids',ip,torch.long),('attention_mask',am,torch.long),
        ('marker_pos',mp,torch.long),('marker_mask',mm,torch.bool),('qtype',qt,torch.long)]}

def prepare_batches(items,batch_size):
    return [make_batch(items[i:i+batch_size]) for i in range(0,len(items),batch_size)]

CAL_BATCHES=prepare_batches(cal_items,SELECTION_BATCH_SIZE)
SEL_BATCHES=prepare_batches(selection_items,SELECTION_BATCH_SIZE)
AUDIT_BATCHES=prepare_batches(audit_items,SELECTION_BATCH_SIZE)
print('Case-disjoint decision counts:',len(cal_items),len(selection_items),len(audit_items))
print('Gold coverage selection',sum(r['gold'] is not None for r in selection_items),'/',len(selection_items))

In [ ]:
#@title 7. Baseline model outputs, true task labels, and speed
@torch.inference_mode()
def infer(m,b):
    with torch.autocast('cuda',dtype=torch.float16):
        return m(**b)

@torch.inference_mode()
def capture(m,batches):
    collected=[]
    for b in batches:
        scores,acts=infer(m,b)
        probs=torch.softmax(scores.float().masked_fill(~b['marker_mask'],-1e4),-1)
        collected.append((probs.cpu(),acts.float().cpu()))
    return collected

SEL_BASE=capture(model,SEL_BATCHES)
AUDIT_BASE=capture(model,AUDIT_BATCHES)

def evaluate(m,batches,reference,items):
    total=correct=gold_count=gold_correct=0
    js_total=action_mse=0.
    with torch.inference_mode():
        for (rprob,ract),b in zip(reference,batches):
            scores,act=infer(m,b)
            probs=torch.softmax(scores.float().masked_fill(~b['marker_mask'],-1e4),-1).cpu()
            act=act.float().cpu()
            n=probs.shape[0]
            offset=total
            total+=n
            pred=probs.argmax(-1)
            correct+=int((pred==rprob.argmax(-1)).sum())
            middle=0.5*(probs+rprob)
            divergences=0.5*(probs*(probs.clamp_min(1e-9).log()-middle.clamp_min(1e-9).log())).sum(-1)
            divergences+=0.5*(rprob*(rprob.clamp_min(1e-9).log()-middle.clamp_min(1e-9).log())).sum(-1)
            js_total+=float(divergences.sum())
            action_mse+=float((act-ract).square().mean())*n
            for i in range(n):
                g=items[offset+i]['gold']
                if g is not None:
                    gold_count+=1
                    gold_correct+=int(int(pred[i])==g)
    return dict(n=total,decision_agreement=correct/total,mean_js=js_total/total,
                action_mse=action_mse/total,
                gold_n=gold_count,gold_accuracy=(gold_correct/gold_count if gold_count else None))

# Batches for speed: fixed real input lengths and shapes, same before/after quantization.
BENCH_BATCHES={}
for batch_size in [1,4]:
    for seq_len in [128,256,512]:
        if seq_len>MAX_LEN:continue
        chosen=selection_items[:batch_size]
        try:BENCH_BATCHES[f'B{batch_size}_T{seq_len}']=make_batch(chosen,pad_to=seq_len)
        except ValueError:pass
assert BENCH_BATCHES

@torch.inference_mode()
def benchmark(m,b,warmup=3,repeats=BENCH_REPEATS):
    for _ in range(warmup):infer(m,b)
    torch.cuda.synchronize()
    ticks=[]
    for i in range(repeats):
        torch.cuda.synchronize();start=time.perf_counter()
        infer(m,b)
        torch.cuda.synchronize();ticks.append(1000*(time.perf_counter()-start))
    return float(np.median(ticks))

BASE_BENCH={name:benchmark(model,b) for name,b in BENCH_BATCHES.items()}
BASE_SELECTION=evaluate(model,SEL_BATCHES,SEL_BASE,selection_items)
BASE_AUDIT=evaluate(model,AUDIT_BATCHES,AUDIT_BASE,audit_items)
BASE_GOLD=BASE_SELECTION['gold_accuracy']
print('Original precision reference:',BASE_SELECTION,'latency ms:',BASE_BENCH)

In [ ]:
#@title 8. Calibrate input channel energy for each projection (per-layer, no training)
@torch.inference_mode()
def activation_importance(m,module_names,cal_batches):
    if CALIBRATION_METHOD!='activation_mse':return {}
    accum={}; counts={}; hooks=[]
    for name in module_names:
        src=m.get_submodule(name)
        def collect(module,args,name=name):
            x=args[0].detach().float()
            values=x.reshape(-1,x.shape[-1])
            sums=values.square().sum(dim=0)
            accum[name]=accum.get(name,0)+sums
            counts[name]=counts.get(name,0)+values.shape[0]
        hooks.append(src.register_forward_pre_hook(collect))
    try:
        for batch in cal_batches:infer(m,batch)
    finally:
        for h in hooks:h.remove()
    return {name:(accum[name]/counts[name]).clamp_min(1e-8) for name in accum}

def projection_variants(names,m):
    attn=[n for n in names if '.attn.' in n]
    ffn=[n for n in names if '.attn.' not in n]
    candidates=[('all',names),('attention',attn),('feed_forward',ffn)]
    biggest=max(names,key=lambda n:m.get_submodule(n).weight.numel())
    candidates.append(('single_largest',[biggest]))
    seen=set();unique=[]
    for title,group in candidates:
        key=tuple(group)
        if not group or key in seen:continue
        seen.add(key);unique.append((title,group))
    return unique

def gate(metrics):
    # The cumulative candidate is compared with the ORIGINAL model on selection data.
    okay=(metrics['decision_agreement']>=MIN_AGREEMENT and metrics['mean_js']<=MAX_MEAN_JS)
    if BASE_GOLD is not None and metrics['gold_accuracy'] is not None:
        okay=okay and metrics['gold_accuracy']>=BASE_GOLD-MAX_GOLD_ACCURACY_DROP
    return bool(okay)

print('Gate min agreement:',MIN_AGREEMENT,'max JS:',MAX_MEAN_JS,
      'max gold accuracy drop:',MAX_GOLD_ACCURACY_DROP)

In [ ]:
#@title 9. Main one-encoder-layer-at-a-time trial (COMMIT / ROLLBACK)
assert 0<=FIRST_LAYER<LAST_LAYER_EXCLUSIVE<=len(LAYER_PATHS)
order=list(range(FIRST_LAYER,LAST_LAYER_EXCLUSIVE))
if LAYER_ORDER=='reverse':order.reverse()
trial_rows=[];accept_rows=[];accepted_names=[]
for layer_i in order:
    names=LAYER_PATHS[layer_i]
    if not names:continue
    print('\n'+'='*90)
    print(f'LAYER {layer_i:02d}, {len(names)} candidate matrices — collecting calibration activations')
    importance=activation_importance(model,names,CAL_BATCHES)
    originals={n:model.get_submodule(n) for n in names}
    replacements={n:g128.TernaryG128Linear.from_linear(originals[n],
                      importance=importance.get(n),method=CALIBRATION_METHOD) for n in names}
    variants=projection_variants(names,model)
    passed=[]
    for variant,subset in variants:
        # Make each trial independently: never inadvertently leave a failed replacement in model.
        for name in subset:g128.swap(model,name,replacements[name])
        try:
            q=evaluate(model,SEL_BATCHES,SEL_BASE,selection_items)
            accepted=gate(q)
            # Original linear includes bias; subtract packed module bytes for exact delta.
            savings=sum(g128.state_bytes(originals[n])-g128.state_bytes(replacements[n]) for n in subset)
            row={'layer':layer_i,'variant':variant,'modules':len(subset),
                 'weight_count':sum(originals[n].weight.numel() for n in subset),
                 'saved_mib':savings/2**20,'pass':accepted,'method':CALIBRATION_METHOD,**q}
            trial_rows.append(row)
            if accepted:passed.append((savings,-q['mean_js'],variant,subset,q))
            print(f'{variant:15s}: agreement={q["decision_agreement"]:.4f}, JS={q["mean_js"]:.5f}, '
                  f'saved={savings/2**20:.2f}MiB | {"PASS" if accepted else "REJECT"}')
        except Exception as exc:
            trial_rows.append({'layer':layer_i,'variant':variant,'pass':False,'error':repr(exc)})
            print('FAILED:',variant,repr(exc)[:320])
        finally:
            for name in subset:g128.swap(model,name,originals[name])
    if passed:
        best=max(passed,key=lambda r:(r[0],r[1]))
        _,_,chosen,subset,_=best
        for name in subset:g128.swap(model,name,replacements[name])
        accepted_names.extend(subset)
        status='COMMIT '+chosen
    else:
        chosen=None;subset=[];status='ROLLBACK'
    gc.collect()
    # Persist diagnostics after EACH layer so interrupted Colab runs retain evidence.
    current=evaluate(model,SEL_BATCHES,SEL_BASE,selection_items)
    rec={'layer':layer_i,'status':status,'variant':chosen,'converted_modules_total':len(accepted_names),
         'converted_weights_total':sum(model.get_submodule(n).out_features*model.get_submodule(n).in_features for n in accepted_names),
         'packed_state_mib':g128.state_bytes(model)/2**20,**current}
    accept_rows.append(rec)
    pd.DataFrame(trial_rows).to_csv(OUTPUT/'layer_trials.csv',index=False)
    pd.DataFrame(accept_rows).to_csv(OUTPUT/'layerwise_cumulative.csv',index=False)
    (OUTPUT/'accepted_modules.json').write_text(json.dumps(accepted_names,indent=2))
    print(f'{status} | running total: {len(accepted_names)} matrices | '
          f'agreement={current["decision_agreement"]:.3%}, state={rec["packed_state_mib"]:.1f}MiB')

trials=pd.DataFrame(trial_rows)
progress=pd.DataFrame(accept_rows)
display(progress)
print('Accepted modules:',len(accepted_names),'/',sum(len(l) for l in LAYER_PATHS))

In [ ]:
#@title 10. Independent held-out audit, actual end-to-end GPU benchmark, layerwise charts
FINAL_SELECT=evaluate(model,SEL_BATCHES,SEL_BASE,selection_items)
FINAL_AUDIT=evaluate(model,AUDIT_BATCHES,AUDIT_BASE,audit_items)
FINAL_BENCH={name:benchmark(model,b) for name,b in BENCH_BATCHES.items()}
FINAL_BYTES=g128.state_bytes(model)
SELECT_GATE=gate(FINAL_SELECT)
AUDIT_GATE=(FINAL_AUDIT['decision_agreement']>=MIN_AGREEMENT and FINAL_AUDIT['mean_js']<=MAX_MEAN_JS)
if BASE_AUDIT['gold_accuracy'] is not None and FINAL_AUDIT['gold_accuracy'] is not None:
    AUDIT_GATE=AUDIT_GATE and FINAL_AUDIT['gold_accuracy']>=BASE_AUDIT['gold_accuracy']-MAX_GOLD_ACCURACY_DROP
FINAL_GATE=bool(SELECT_GATE and AUDIT_GATE and len(accepted_names)>0)
SPEEDUPS={k:BASE_BENCH[k]/max(v,1e-9) for k,v in FINAL_BENCH.items()}
report={'source_repo':SOURCE_REPO,'method':CALIBRATION_METHOD,'weight_format':'bonsai_inspired_q2_g128_v1',
        'group_size':128,'codes':[-1,0,1],'bits_per_weight_including_group_scale':2.125,
        'original_params':TOTAL_PARAMS,'eligible_linear_weights':ALL_ELIGIBLE,
        'selected_module_names':accepted_names,'original_state_bytes':ORIGINAL_STATE_BYTES,
        'final_state_bytes':FINAL_BYTES,'state_reduction_ratio':ORIGINAL_STATE_BYTES/FINAL_BYTES,
        'baseline_gold':BASE_GOLD,'selection':FINAL_SELECT,'audit':FINAL_AUDIT,
        'baseline_cuda_wall_ms':BASE_BENCH,'final_cuda_wall_ms':FINAL_BENCH,
        'actual_end_to_end_speedups':SPEEDUPS,'selection_gate_pass':SELECT_GATE,
        'audit_gate_pass':AUDIT_GATE,'final_gate_pass':FINAL_GATE,
        'selection_and_audit_from_train_case_disjoint':True,'official_test_used_for_selection':False}
(OUTPUT/'summary.json').write_text(json.dumps(report,indent=2))
print(json.dumps({k:report[k] for k in ['selection','audit','state_reduction_ratio','actual_end_to_end_speedups','final_gate_pass']},indent=2))

import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,2,figsize=(13,4))
if len(progress):
    ax[0].plot(progress.layer,progress.decision_agreement,marker='.')
    ax[0].axhline(MIN_AGREEMENT,linestyle='--',label='Selection gate')
    ax[0].set(title='Cumulative agreement with original model',xlabel='Encoder layer',ylabel='Agreement',ylim=(0,1.01))
    ax[0].legend()
    ax[1].plot(progress.layer,progress.packed_state_mib,marker='.')
    ax[1].set(title='Actual state_dict storage in memory',xlabel='Encoder layer',ylabel='MiB')
fig.tight_layout()
fig.savefig(OUTPUT/'layerwise_results.png',dpi=180)
plt.show()

In [ ]:
#@title 11. Optional untouched official test (only AFTER method/layers are fixed)
if EVALUATE_OFFICIAL_TEST:
    test_rows=list(load_dataset('LocalLLaMA/typed-decisions','all',split='test'))
    test_items=items_from_rows(test_rows)
    test_batches=prepare_batches(test_items,SELECTION_BATCH_SIZE)
    # Reference model is reloaded to avoid mixing converted activations into baseline.
    teacher,_=original_loader.load_model(ROOT,device='cpu')
    teacher.to('cuda',dtype=torch.float16).eval().requires_grad_(False)
    test_ref=capture(teacher,test_batches)
    test_result=evaluate(model,test_batches,test_ref,test_items)
    teacher_gold_num=teacher_gold_ok=0
    offset=0
    for ref_prob,_ in test_ref:
        for i in range(len(ref_prob)):
            g=test_items[offset+i]['gold']
            if g is not None:
                teacher_gold_num+=1;teacher_gold_ok+=int(int(ref_prob[i].argmax())==g)
        offset+=len(ref_prob)
    report['official_test']={'compressed':test_result,
        'original_gold_accuracy':(teacher_gold_ok/teacher_gold_num if teacher_gold_num else None),
        'n':len(test_items),'used_for_selection':False}
    (OUTPUT/'summary.json').write_text(json.dumps(report,indent=2))
    print(json.dumps(report['official_test'],indent=2))
    del teacher;gc.collect();torch.cuda.empty_cache()
else:
    print('Official test intentionally untouched; enable EVALUATE_OFFICIAL_TEST after selecting architecture.')

In [ ]:
#@title 12. Export an independent, packed, strict-reloadable Jev_CeNN checkpoint
from safetensors.torch import save_file
BUNDLE=OUTPUT/'bonsai_g128_bundle'
if not FINAL_GATE:
    print('Export blocked: selection and independent audit gates not both passing.')
else:
    BUNDLE.mkdir(exist_ok=True,parents=True)
    for folder in ['encoder','tokenizer']:
        shutil.copytree(ROOT/folder,BUNDLE/folder,dirs_exist_ok=True)
    for filename in ['runtime.py','bundle.json']:
        shutil.copy2(ROOT/filename,BUNDLE/filename)
    for filename in ['jev_bonsai_g128_runtime.py','jev_bonsai_g128_loader.py']:
        shutil.copy2('/content/'+filename,BUNDLE/filename)
    modules=[{'name':n,'input_features':model.get_submodule(n).in_features,
              'output_features':model.get_submodule(n).out_features} for n in accepted_names]
    meta={'format':'bonsai_inspired_q2_g128_v1','source':SOURCE_REPO,'group_size':128,
          'quantized_modules':modules,'report_file':'summary.json','not_original_bonsai_training':True}
    (BUNDLE/'bonsai_meta.json').write_text(json.dumps(meta,indent=2))
    shutil.copy2(OUTPUT/'summary.json',BUNDLE/'summary.json')
    state={n:t.detach().contiguous().cpu().clone() for n,t in model.state_dict().items()}
    save_file(state,str(BUNDLE/'model_bonsai.safetensors'))
    del state
    (BUNDLE/'requirements.txt').write_text('torch>=2.4\ntransformers==4.57.6\nsafetensors>=0.4\nhuggingface_hub>=0.25,<1\ntriton>=3.0,<4\n')
    (BUNDLE/'README.md').write_text('# Jev_CeNN g128 (Bonsai-inspired)\n\n'
       'One-group-scale-per-128-weight ternary post-training quantization experiment.\n'
       'Not an implementation of unpublished Bonsai quantization/training algorithms.\n'
       'Requires trusted custom runtime and Triton CUDA kernel.\n'
       'To load: `from jev_bonsai_g128_loader import load_bonsai; model,tokenizer,runtime=load_bonsai(PATH)`\n'
       'Review summary.json for retained modules and quality/latency measurements.\n')
    import jev_bonsai_g128_loader as qload
    fresh,_,_=qload.load_bonsai(BUNDLE,device='cuda')
    for b in SEL_BATCHES[:3]:
        lhs=infer(model,b);rhs=infer(fresh,b)
        for x,y in zip(lhs,rhs):torch.testing.assert_close(x.float(),y.float(),atol=0.012,rtol=0.012)
    print('Strict state_dict reload/output parity: PASS | packed file MiB:',
          round((BUNDLE/'model_bonsai.safetensors').stat().st_size/2**20,2))
    del fresh;gc.collect();torch.cuda.empty_cache()

In [ ]:
#@title 13. Optional publish a separate PRIVATE checkpoint; original source is never overwritten
if not PUBLISH_TO_HF:
    print('PUBLISH_TO_HF=False: no upload. Original private model is unchanged.')
elif not FINAL_GATE:
    print('Publication blocked because selection/audit gates did not both pass.')
else:
    from huggingface_hub import HfApi
    api=HfApi(token=HF_TOKEN)
    assert OUTPUT_REPO!=SOURCE_REPO,'Cannot overwrite the original checkpoint.'
    api.create_repo(OUTPUT_REPO,repo_type='model',private=True,exist_ok=True)
    assert api.repo_info(OUTPUT_REPO,repo_type='model').private, 'Refusing to publish into a public repo.'
    result=api.upload_folder(repo_id=OUTPUT_REPO,folder_path=str(BUNDLE),repo_type='model',
                             commit_message='Add audited g128 Bonsai-inspired layerwise ternary checkpoint')
    print('Private model published:',result.commit_url)

In [ ]:
#@title 14. Export ablation results to zip for review
import zipfile
z=OUTPUT/'Jev_CeNN_Bonsai_G128_Results.zip'
with zipfile.ZipFile(z,'w',zipfile.ZIP_DEFLATED) as archive:
    for filename in ['layer_trials.csv','layerwise_cumulative.csv','accepted_modules.json',
                     'summary.json','layerwise_results.png']:
        if (OUTPUT/filename).is_file():archive.write(OUTPUT/filename,filename)
print('Results archive:',z)
try:
    from google.colab import files
    files.download(str(z))
except ImportError:
    print('Files are saved in:',OUTPUT)